You are evaluating two candidate sentiment models against the sentiment model already running in production (the **baseline**). Your goal is to decide whether either candidate should ship.

“Ship” means: we would turn off the baseline and serve the candidate instead, based on the evidence you collect. Read `README.md` first — it defines the five concepts this lab is built on and lists your deliverables.

### Step 1 - Install the required dependencies, set up W&B and make sure the python version is 3.10 and above

In [ ]:
# Run once. Skip this cell on later runs — everything is already installed.
!pip install -q wandb datasets transformers torch tqdm emoji pandas pyarrow scikit-learn pytest pyyaml

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import wandb

from datasets import load_dataset
from transformers import pipeline

SEED = 42
np.random.seed(SEED)

In [ ]:
# Run `wandb login` in a terminal first; this just confirms the notebook sees your key.
wandb.login()

In [ ]:
!python --version

In [ ]:
# W&B config — every run logged below reuses these three values.
PROJECT = "mlip-lab4-slices-2026"
ENTITY = None  # None = your own account; set a team name only if your TA gives you one
RUN_NAME = "baseline_vs_candidate"


In [ ]:
# Set True in Task G, after you freeze tests/manifest.yaml. Do not retune the gate for v2.
INCLUDE_CANDIDATE_V2 = False

# The registry lives in lab_helpers.py so this notebook and the pytest gate can never
# disagree about which model is which. `csv_col` is that model's column prefix in tweets.csv.
from lab_helpers import BASELINE, MODELS as MODEL_REGISTRY

MODELS = {
    name: spec
    for name, spec in MODEL_REGISTRY.items()
    if INCLUDE_CANDIDATE_V2 or name != "candidate_v2"
}
pd.DataFrame(MODELS).T

In [ ]:
# Label normalization for tweet_eval (0/1/2 -> string labels)
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}

# Many HF sentiment models output labels like LABEL_0 / LABEL_1 / LABEL_2
HF_LABEL_MAP = {
    "LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive",
    "NEG": "negative", "NEU": "neutral", "POS": "positive",
    "0": "negative", "1": "neutral", "2": "positive",
}

USE_HF_DATASET = False  # set True to load tweet_eval from Hugging Face and run inference

### Step 2 - Load a dataset from Hugging Face

In [ ]:
if USE_HF_DATASET:
    ds = load_dataset("cardiffnlp/tweet_eval", "sentiment")
    df = pd.DataFrame(ds["test"]).head(500).copy()
    df["label"] = df["label"].map(ID2LABEL)
    df = df[["text", "label"]].dropna().reset_index(drop=True)
else:
    df = pd.read_csv("tweets.csv")
    # Keep prediction columns (roberta, gpt2, …) so Step 4 can skip live inference.
    df = df.rename(columns={c: c.strip() for c in df.columns})
    assert {"text", "label"}.issubset(df.columns), "tweets.csv must include text,label"
    df["label"] = df["label"].astype(str).str.lower()
    df = df.dropna(subset=["text", "label"]).reset_index(drop=True)

df.head(3)


### Step 3 - Define Failure-Relevant Metadata

#TODO:
In this step, you will create **at least 5** metadata columns that help you slice and analyze model behavior in Weights & Biases (W&B).
These metadata columns should **capture meaningful properties of the data or model behavior that may influence performance**. You can define them using:

1. Value matching (e.g., tweets containing hashtags or mentions)
2. Regex patterns (e.g., negation words, strong sentiment terms like love or hate)
3. Heuristics (e.g., emoji count, all-caps text, tweet length buckets)

Each metadata column should correspond to a potential hypothesis about when or why a model might succeed or fail.
These columns will be propagated through inference and included in the final predictions_table logged to W&B.

After inference, your W&B table (df_long) will contain:
- The original tweet text
- Ground-truth sentiment labels
- Model predictions and confidence scores
- All metadata columns you defined for slicing

You will use these metadata fields in the W&B UI (via the ➕ Filter option) to:
- Create slices of the data
- Compare model behavior across slices
- Identify patterns, weaknesses, or regressions that are not visible in overall accuracy

In [ ]:
# Step 3 – Add slicing metadata (text-only)
#
# TODO: add your own hypothesis-driven metadata here.
# Edit slices.py (META_COLS, add_metadata, and get_slices). Here are examples
# of the kinds of metadata columns you can add & analyse.
# Categories you can explore are: Linguistic, Emotional/semantic, Model-behavioral.
# Do not reuse the ones given below.

from slices import META_COLS, add_metadata, get_slices

assert len(META_COLS) >= 5
df = add_metadata(df)
df[META_COLS].head(3)

In [ ]:
# Transformers requires a backend (PyTorch/TensorFlow/Flax). We'll use PyTorch.
try:
    import torch, transformers, sys
    print("torch:", torch.__version__)
    print("transformers:", transformers.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Python:", sys.executable)
except Exception as e:
    raise RuntimeError("Install PyTorch before proceeding: pip install torch torchvision torchaudio") from e

###  Step 4 – Run Inference (Models in MODELS)

In this step, you'll score every model in `MODELS`. With `USE_HF_DATASET = False` (the default), predictions are read from `tweets.csv`. Set it to `True` to run HuggingFace inference instead.

In [ ]:
from tqdm.auto import tqdm

def run_pipeline(model_id: str, texts: list[str]):
    clf = pipeline(
        "text-classification",
        model=model_id,
        truncation=True,
        max_length=128,     # avoid truncation warnings
        framework="pt",
        device=-1           # CPU
    )
    # (Optional) sanity check label mapping for this model
    # print(model_id, clf.model.config.id2label)

    preds, confs = [], []
    for t in tqdm(texts, desc=f"Infer: {model_id}"):
        out = clf(t)[0]
        lbl = HF_LABEL_MAP.get(out["label"], out["label"])
        preds.append(lbl)
        confs.append(float(out["score"]))
    return preds, confs

pred_frames = []

if not USE_HF_DATASET:
    # Build df_long from cached columns in tweets.csv (no network).
    for model_name, spec in MODELS.items():
        col = spec["csv_col"]
        score_col = f"{col}_score"
        if col not in df.columns or score_col not in df.columns:
            print(f"Skipping {model_name}: tweets.csv missing {col}/{score_col}")
            continue
        keep = ["text", "label"] + [c for c in META_COLS if c in df.columns]
        tmp = df[keep].copy()
        tmp["model"] = model_name
        tmp["pred"] = df[col].astype(str).str.lower()
        tmp["conf"] = pd.to_numeric(df[score_col], errors="coerce")
        lat_col = f"{col}_latency_ms"
        if lat_col in df.columns:
            tmp["latency_ms"] = pd.to_numeric(df[lat_col], errors="coerce")
        pred_frames.append(tmp)
else:
    texts = df["text"].tolist()
    for model_name, spec in MODELS.items():
        yhat, conf = run_pipeline(spec["hf_id"], texts)
        tmp = df.copy()
        tmp["model"] = model_name
        tmp["pred"] = yhat
        tmp["conf"] = conf
        pred_frames.append(tmp)

df_long = pd.concat(pred_frames, ignore_index=True)

# Add a stable example id so reshaping won't silently drop duplicates
df_long["ex_id"] = df_long.groupby(["text", "label"]).ngroup()

df_long.head(5)

In [ ]:
# Step 4.5 – Wide-format Table for Model Comparison (Optional but recommended)
# One row per tweet, with each model's predictions in columns
# TODO: Replace with your metadata
assert len(META_COLS) >= 5
df_wide = df_long.pivot_table(
    index=["ex_id", "text", "label"] + META_COLS,
    columns="model",
    values=["pred", "conf"],
    aggfunc="first"
).reset_index()

# Flatten column names (e.g., pred_baseline, conf_candidate_v1)
df_wide.columns = ["_".join([c for c in col if c]).strip("_") for col in df_wide.columns]

df_wide.head(5)

### Step 5: Compute Metrics (Accuracy + Slice Accuracy + Regression)

In [ ]:
# TODO: Edit to work for your slices

#compute metrics model-wise
from sklearn.metrics import accuracy_score

def compute_accuracy(y_true, y_pred):
    return accuracy_score(list(y_true), list(y_pred))

# Overall accuracy by model (df_long: one row per (tweet, model))
overall = df_long.groupby("model").apply(
    lambda g: compute_accuracy(g["label"], g["pred"]),
    include_groups=False
)

# Slice accuracy table (uses df_long masks)
slice_table = wandb.Table(columns=["slice", "model", "accuracy"])
slice_metrics = {}

for slice_name, mask in get_slices(df_long).items():
    slice_metrics[slice_name] = {}
    for model_name, g in df_long[mask].groupby("model"):
        acc = float(compute_accuracy(g["label"], g["pred"]))
        slice_table.add_data(slice_name, model_name, acc)
        slice_metrics[slice_name][model_name] = acc

In [ ]:
# TODO: Edit to work for your slices


# Regression-aware evaluation (df_eval: one row per tweet, all model outputs)
# A regression is when a candidate gets something wrong that the baseline got right.
# BASELINE is defined with MODELS in Step 1.

# Ensure ex_id exists (safe even if it already exists)
df_long = df_long.copy()
if "ex_id" not in df_long.columns:
    df_long["ex_id"] = df_long.groupby(["text", "label"]).ngroup()

assert len(META_COLS) >= 5
df_eval = (
    df_long.pivot_table(
        index=["ex_id", "text", "label"] + META_COLS,
        columns="model",
        values=["pred", "conf"],
        aggfunc="first"
    )
    .reset_index()
)

# Flatten column names (pred_baseline, conf_candidate_v1, etc.)
df_eval.columns = ["_".join([c for c in col if c]).strip("_") for col in df_eval.columns]

if f"pred_{BASELINE}" not in df_eval.columns:
    raise KeyError(f"baseline predictions missing from df_eval (need pred_{BASELINE})")

CANDIDATES = [
    m for m in MODELS
    if m != BASELINE and f"pred_{m}" in df_eval.columns
]

df_eval["baseline_correct"] = df_eval[f"pred_{BASELINE}"] == df_eval["label"]

regression_rate = {}
improvement_rate = {}
conf_reg_rate = {}

for cand in CANDIDATES:
    df_eval[f"{cand}_correct"] = df_eval[f"pred_{cand}"] == df_eval["label"]
    df_eval[f"{cand}_regressed"] = df_eval["baseline_correct"] & ~df_eval[f"{cand}_correct"]
    df_eval[f"{cand}_improved"] = ~df_eval["baseline_correct"] & df_eval[f"{cand}_correct"]
    df_eval[f"{cand}_both_wrong"] = ~df_eval["baseline_correct"] & ~df_eval[f"{cand}_correct"]
    df_eval[f"{cand}_both_correct"] = df_eval["baseline_correct"] & df_eval[f"{cand}_correct"]
    df_eval[f"{cand}_confident_regression"] = (
        df_eval[f"{cand}_regressed"] & (df_eval[f"conf_{cand}"] >= 0.8)
    )
    regression_rate[cand] = float(df_eval[f"{cand}_regressed"].mean())
    improvement_rate[cand] = float(df_eval[f"{cand}_improved"].mean())
    conf_reg_rate[cand] = float(df_eval[f"{cand}_confident_regression"].mean())
    print(cand, "regression rate:", regression_rate[cand])
    print(cand, "improvement rate:", improvement_rate[cand])
    print(cand, "confident regression rate:", conf_reg_rate[cand])

In [ ]:
# TODO: Edit to work with your slices

# Define slices on df_eval (must use columns that exist in df_eval)
def get_slices_eval(df_any):
    slices = dict(get_slices(df_any))
    slices["long_tweets"] = df_any["length_bucket"].astype(str).isin(["201-1000", "1001+"])
    return slices

# Slice-level regression metrics table (one row per slice × candidate × metric)
reg_table = wandb.Table(columns=["slice", "candidate", "metric", "value"])
reg_metrics = {}

for slice_name, mask in get_slices_eval(df_eval).items():
    g = df_eval[mask]
    if len(g) == 0:
        continue

    reg_metrics[slice_name] = {}
    for cand in CANDIDATES:
        reg = float(g[f"{cand}_regressed"].mean())
        imp = float(g[f"{cand}_improved"].mean())
        conf_reg = float(g[f"{cand}_confident_regression"].mean())

        reg_table.add_data(slice_name, cand, "regression_rate", reg)
        reg_table.add_data(slice_name, cand, "improvement_rate", imp)
        reg_table.add_data(slice_name, cand, "confident_regression_rate", conf_reg)

        reg_metrics[slice_name][cand] = {
            "regression_rate": reg,
            "improvement_rate": imp,
            "conf_reg_rate": conf_reg
        }

# Step 6 — #TODO: Log to W&B & Analyse Slices
# (Make sure PROJECT/ENTITY/RUN_NAME exist from Step 1)

In [ ]:
# Step 6: Log to W&B (PROJECT / ENTITY / RUN_NAME come from Step 1)

run = wandb.init(project=PROJECT, entity=ENTITY, name=RUN_NAME)
wandb.log({"predictions_table": wandb.Table(dataframe=df_long)})
wandb.log({"slice_metrics": slice_table})
wandb.log({"regression_metrics": reg_table})
wandb.log({
    "df_eval": wandb.Table(dataframe=df_eval)
})
for model_name, acc in overall.items():
    wandb.summary[f"{model_name}_accuracy"] = float(acc)
for cand, rate in regression_rate.items():
    wandb.summary[f"{cand}_regression_rate"] = rate
    wandb.summary[f"{cand}_improvement_rate"] = improvement_rate[cand]
    wandb.summary[f"{cand}_confident_regression_rate"] = conf_reg_rate[cand]

print("W&B run URL:", getattr(run, "url", None) or getattr(run, "get_url", lambda: None)())
run.finish()

### Instructions: Exploring Slice-Based Evaluation in W&B

# Purpose
In this lab, you are evaluating a candidate sentiment model to decide whether it should replace an existing baseline (production) model.
You have already:
  - scored every model in MODELS on the same 500 tweets
  - logged predictions, confidence scores, and metadata to W&B
  - created metadata that allows you to slice the data
The most important goal is to understand when and why models behave differently.
Overall accuracy alone is often misleading.

# What to do in W&B
1. Open your W&B run
  - Click the project link and open the latest run.
2. Explore the predictions table
  - Go to the Tables tab and open predictions_table.
  - Each row is one tweet × one model.
3. Create and analyze slices (most important)
  - Use filters to create meaningful slices 
    (e.g., negation, emojis, hashtags, long tweets).
  - For each slice:
    - Compare baseline vs candidate performance.
    - Compare slice accuracy to overall accuracy.
    - Inspect a few misclassified examples to identify patterns.
4. Visualize slice performance
  - Open slice_metrics.
  - Create bar charts comparing baseline vs candidate accuracy for at least two slices.
5. Discuss your findings with the TA
  - Explain why slicing reveals issues that overall accuracy hides.
  - Say whether the candidate model should be deployed and why.


In [ ]:
# Students: replace the placeholders below with 1–2 sentence insights
#TODO: Replace this with 1-2 sentence takeaways for each slice.
saved_slice_notes = ["..."]
pd.DataFrame(saved_slice_notes)

### Step 7 - Targeted stress testing with LLMs

TODO: 
In this step, you will use a Large Language Model (LLM) to generate test cases that specifically target a weakness you observed during slicing.

What to do:
1. Choose one slice where you noticed poor performance, regressions, or surprising behavior.
2. Write a short hypothesis (1–2 sentences) explaining why the model might struggle on this slice. Example:
“The model struggles with tweets that use slang and sarcasm.”
3. Use an LLM to generate 10 test cases designed to test this hypothesis.
These can include:
    - subtle or ambiguous cases
    - difficult or adversarial cases
    - small wording changes that affect sentiment
4. Re-run every model in `MODELS` on the generated test cases (helper code given below).
5. Briefly describe what you observed to the TA:
    - Did the same failures appear again?
    - notice any new failure patterns?
    - would this affect your confidence in deploying the model?

Your input can be in the following format:

> Examples:
> - @user @user That’s coming, but I think the victims are going to be Medicaid recipients.
> - I think I may be finally in with the in crowd #mannequinchallenge  #grads2014 @user
> 
> Generate more tweets using slangs.

Use our provided GPTs to start the task: [llm-based-test-case-generator](https://chatgpt.com/g/g-982cylVn2-llm-based-test-case-generator). If you do not have access to GPTs, use the plain ChatGPT or other LLM providers you have access to instead.

**This is the one step that needs internet.** Steps 1–6 read saved predictions, but your tweets are new, so the cell below downloads each model in `MODELS` and runs it — about 1 GB for `baseline` + `candidate_v1`, on the first run only. Do this step while `INCLUDE_CANDIDATE_V2 = False`. If you come back and re-run it after Task G, it will also pull `candidate_v2`, which is a **1.7 GB** download and roughly 6× slower per tweet.

In [ ]:
# TODO: Paste your 10 generated tweets here:
generated_slice_description = ""

generated_cases = [""] * 10
intended_labels = [""] * 10
assert len(generated_cases) == 10 and len(intended_labels) == 10

In [ ]:
#Helper code to run models on synthetic test cases:

def run_on_generated_tests(texts, models=MODELS):
    rows = []
    for model_name, spec in models.items():
        hf_id = spec["hf_id"] if isinstance(spec, dict) else spec
        clf = pipeline(
            "text-classification",
            model=hf_id,
            truncation=True,
            framework="pt",
            device=-1
        )
        for t in texts:
            out = clf(t)[0]
            rows.append({
                "text": t,
                "model": model_name,
                "pred": HF_LABEL_MAP.get(out["label"], out["label"]),
                "conf": float(out["score"])
            })
    return pd.DataFrame(rows)


In [ ]:
# Skip live HF inference until 10 real tweets are pasted (keeps the notebook offline-runnable).
if not any(str(t).strip() for t in generated_cases):
    generated_df = pd.DataFrame(columns=["text", "model", "pred", "conf", "intended", "correct"])
    print("TODO: paste 10 generated tweets and intended labels, then re-run this cell.")
else:
    generated_df = run_on_generated_tests(generated_cases)
    intended = pd.DataFrame({"text": generated_cases, "intended": intended_labels})
    generated_df = generated_df.merge(intended, on="text", how="left")
    generated_df["correct"] = generated_df["pred"] == generated_df["intended"]
    print(generated_slice_description)
    print(generated_df.groupby("model")["correct"].mean())
generated_df

In [ ]:
# OPTIONAL: Log synthetic test cases to W&B
if len(generated_df):
    synth_run = wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name="synthetic-tests",
        job_type="stress-test",
    )
    wandb.log({"synthetic_tests": wandb.Table(dataframe=generated_df)})
    print("W&B run URL:", getattr(synth_run, "url", None))
    synth_run.finish()
else:
    print("Skipping W&B log: no synthetic rows yet.")

### Step 8 — Exploration vs enforcement (regression gate)

Steps 1–7 were **exploration**: you sliced in a notebook, stared at W&B, and generated synthetic tweets. That is how you *find* hypotheses.

This step is **enforcement**. You encode the slices you care about as tests with **thresholds chosen before you score the next candidate**. Freeze `tests/manifest.yaml` using the baseline (and what you learned from `candidate_v1`). Then run the **same** suite on `candidate_v2` without editing the manifest. A slice with fewer than 30 examples is skipped — a rate estimated on 5 tweets is noise, not a gate.

Until Task G, leave `INCLUDE_CANDIDATE_V2 = False` in Step 1 so W&B is baseline vs `candidate_v1` only. Predictions for v2 are already in `tweets.csv`; hiding them is so you do not retune the gate.

Edit `tests/manifest.yaml` first (slice name, threshold, rationale, and the p50 latency cap). Then run pytest from the repo root.

The model runs on a live moderation stream, so the product budget is **p50 under 50 ms per tweet on one CPU thread**. Timings are already measured and saved in `tweets.csv`, so pytest reads them instead of re-timing on your laptop.

Manifest format:

```yaml
slices:
  - slice: has_negation
    threshold: 0.60
    rationale: ""
latency:
  p50_ms_max: 50
  rationale: ""
```


In [ ]:
# Run these from the repo root *after* you freeze thresholds in tests/manifest.yaml.
# Do not retune the manifest to make a candidate pass.

print("MODEL=baseline pytest tests/ -v")
print("MODEL=candidate_v1 pytest tests/ -v")
# Task G — same suite, do not edit the manifest:
print("MODEL=candidate_v2 pytest tests/ -v")

In [ ]:
# Load the pytest session CSV and log it to W&B as gate_results.
from pathlib import Path

model = os.environ.get("MODEL", "candidate_v1")
gate_path = Path(f"tests/gate_results_{model}.csv")
if not gate_path.exists():
    print(f"No {gate_path} yet. Run: MODEL={model} pytest tests/ -v")
    gate_df = pd.DataFrame(columns=["test_id", "observed", "threshold", "outcome"])
else:
    gate_df = pd.read_csv(gate_path)
    gate_run = wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name=f"gate-results-{model}",
        job_type="gate",
    )
    wandb.log({"gate_results": wandb.Table(dataframe=gate_df)})
    print("W&B run URL:", getattr(gate_run, "url", None))
    gate_run.finish()
gate_df